# Issue 4: ES futures price-move events

We use the futures cache from Issue 3 to identify price moves for the stale-quote
study. The main table contains isolated midpoint changes. We also keep the raw
BBO changes and clustered paths so the filtering can be checked.

Run all cells from the project root or its `notebooks/` folder. The default is
`RUN_MODE = 'real'`: missing files raise an error rather than switching to demo
data. This notebook only reads local files; it does not download data or need
an API key. Results are written to `issue4_report/` in the project root.


## Settings

The sample covers Monday 5 October to Friday 9 October 2026. Each date uses
the request interval in its Issue 3 cache manifest. Prices are in ES index points and one futures tick
is 0.25 points. Time thresholds and durations are in integer nanoseconds (ns).
The thresholds below are working choices; the sensitivity table shows how much
the event counts change when we alter the clustering gap.


In [1]:
import sys, subprocess, importlib.util
if sys.version_info < (3, 11):
    raise RuntimeError('Select a Python 3.11 or newer notebook kernel.')
packages = {'numpy':'numpy>=1.26', 'pandas':'pandas>=2.1',
    'databento':'databento>=0.40', 'pyarrow':'pyarrow>=14.0',
    'pytest':'pytest>=7.4', 'IPython':'ipython', 'tzdata':'tzdata'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
    importlib.invalidate_caches()
print('Dependencies available in the current kernel.')


Dependencies available in the current kernel.


In [2]:
import os, json, types, tempfile
from pathlib import Path
from datetime import date
from types import SimpleNamespace
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RUN_MODE = 'real'  # 'real', 'demo', or 'auto'
STUDY_DATES = [date(2026, 10, d) for d in range(5, 10)]
RUN_SENSITIVITY = True
CLUSTER_GAP_NS = 250_000
MAX_CLUSTER_SPAN_NS = 2_000_000
REACTION_WINDOW_NS = 10_000_000
MAX_QUOTE_GAP_NS = 1_000_000_000

ROOT = Path(os.environ.get('ISSUE4_PROJECT_ROOT',Path.cwd())).expanduser().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW_DIR = ROOT / 'data' / 'raw'
OUT_DIR = ROOT / 'issue4_report'
# Defaults used by the functions below.
config = SimpleNamespace(DATASET='GLBX.MDP3', ES_FUTURES_TICK_SIZE=.25,
    CLUSTER_GAP_NS=CLUSTER_GAP_NS, MAX_CLUSTER_SPAN_NS=MAX_CLUSTER_SPAN_NS,
    REACTION_WINDOW_NS=REACTION_WINDOW_NS, MAX_QUOTE_GAP_NS=MAX_QUOTE_GAP_NS,
    ROOT_DIR=ROOT, RAW_DIR=RAW_DIR, PROCESSED_DIR=ROOT/'data'/'processed',
    STUDY_DATES=STUDY_DATES)
if RUN_MODE not in {'auto','real','demo'}:
    raise ValueError('RUN_MODE must be auto, real, or demo')
print('Input cache:',RAW_DIR)
print('Outputs:',OUT_DIR)
print('Thresholds (ns):',CLUSTER_GAP_NS,MAX_CLUSTER_SPAN_NS,REACTION_WINDOW_NS)

print('Study dates:', ', '.join(d.isoformat() for d in STUDY_DATES))


Input cache: /Users/christina/Desktop/Futures/reports/Futures_Project-issue-3-download/data/raw
Outputs: /Users/christina/Desktop/Futures/reports/Futures_Project-issue-3-download/issue4_report
Thresholds (ns): 250000 2000000 10000000
Study dates: 2026-10-05, 2026-10-06, 2026-10-07, 2026-10-08, 2026-10-09


## Event definition

The default price is `(best_bid + best_ask) / 2`. ES outright quotes use a
0.25-point tick. A midpoint can move 0.125 points (half a tick); these changes
are retained, not rounded.

A candidate is a change in either BBO price between valid final states at
successive exchange timestamps. Size-only updates are not price candidates.
Trade (`T`) records carry pre-trade quotes and do not establish new quote states.
At a common timestamp, aggregate to the last non-trade state before measuring
the change. Invalid states and continuity gaps reset the comparison.

Bid/ask-only specifications are available through `EventConfig.price_basis`.
Candidates with no change in the chosen price remain auditable but do not
become selected price moves.


In [3]:
"""ES quote-state changes, bounded clusters and non-overlapping study windows.

The public detector accepts point-denominated MBP-1 DataFrames and has no API
or filesystem side effects. The event definition is documented above.
"""
from dataclasses import asdict, dataclass
import math
import numpy as np
import pandas as pd
BBO = ['bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00']
CANDIDATE_COLUMNS = ['candidate_id', 'instrument_id', 'segment_id', 't0', 'price_before', 'price_after', 'move_points', 'move_ticks', 'direction', 'size_class', 'bid_before', 'bid_after', 'ask_before', 'ask_after', 'bid_move_ticks', 'ask_move_ticks', 'mid_before', 'mid_after', 'is_price_move', 'cluster_id']
CLUSTER_COLUMNS = ['cluster_id', 'instrument_id', 'segment_id', 'cluster_start', 'cluster_end', 't0', 'confirmed_at', 'price_before', 'price_after', 'move_points', 'move_ticks', 'abs_move_ticks', 'direction', 'size_class', 'n_updates', 'n_price_moves', 'gross_move_ticks', 'has_reversal', 'is_round_trip', 'left_linked', 'right_linked', 'left_context_missing', 'right_context_missing', 'next_price_move', 'segment_end', 'observation_end', 'observation_end_reason', 'overlap_within_horizon', 'horizon_complete', 'selected', 'exclusion_reason']

@dataclass(frozen=True)
class EventConfig:
    price_basis: str = 'midpoint'
    tick_size: float = config.ES_FUTURES_TICK_SIZE
    cluster_gap_ns: int = config.CLUSTER_GAP_NS
    max_cluster_span_ns: int = config.MAX_CLUSTER_SPAN_NS
    reaction_window_ns: int = config.REACTION_WINDOW_NS
    max_quote_gap_ns: int = config.MAX_QUOTE_GAP_NS
    drop_reversals: bool = True
    singleton_only: bool = False
    overlap_policy: str = 'truncate'

    def __post_init__(self):
        if self.price_basis not in {'midpoint', 'bid', 'ask'}:
            raise ValueError('price_basis must be midpoint, bid or ask')
        if self.overlap_policy not in {'truncate', 'drop'}:
            raise ValueError('overlap_policy must be truncate or drop')
        if not math.isfinite(self.tick_size) or self.tick_size <= 0:
            raise ValueError('tick_size must be finite and positive')
        for name in ('cluster_gap_ns', 'max_cluster_span_ns', 'reaction_window_ns', 'max_quote_gap_ns'):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
                raise ValueError(f'{name} must be an integer number of nanoseconds')
            minimum = 0 if name == 'cluster_gap_ns' else 1
            if value < minimum or value > pd.Timedelta.max.value:
                raise ValueError(f'{name} is outside the supported range')
        if self.max_cluster_span_ns < self.cluster_gap_ns:
            raise ValueError('max_cluster_span_ns must be at least cluster_gap_ns')

@dataclass
class EventResult:
    quotes: pd.DataFrame
    candidates: pd.DataFrame
    clusters: pd.DataFrame
    events: pd.DataFrame
    diagnostics: dict

def _labels(ticks):
    a = np.abs(np.asarray(ticks, dtype=float))
    return np.select([a == 0, a < 1, a == 1], ['flat', 'sub_tick', 'single_tick'], default='multi_tick')

def _utc_ns(values):
    return pd.to_datetime(values, utc=True, errors='coerce').astype('datetime64[ns, UTC]')


In [4]:
def prepare_quotes(frame: pd.DataFrame, settings: EventConfig=EventConfig()) -> tuple[pd.DataFrame, dict]:
    """Normalize BBO states without bridging invalid records or session gaps.

    Trade records carry a pre-trade BBO in MBP-1; they cannot establish a new
    post-trade quote state. An unchanged trade print therefore creates no event.
    Input order is the source arrival order; decreasing exchange times raise.
    """
    d = frame.copy()
    if 'ts_event' not in d and d.index.name == 'ts_event':
        d = d.reset_index()
    required = {'ts_event', 'instrument_id', *BBO}
    missing = required - set(d.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    d = d.reset_index(drop=True)
    d['_arrival'] = np.arange(len(d))
    d['ts_event'] = _utc_ns(d.ts_event)
    ids = pd.to_numeric(d.instrument_id, errors='coerce')
    if ids.isna().any() or not np.isfinite(ids).all() or (ids % 1 != 0).any():
        raise ValueError('instrument_id must contain finite integer identifiers')
    d['instrument_id'] = ids.astype('int64')
    if d.instrument_id.nunique() > 1:
        raise ValueError('Pass one selected underlying future at a time; do not mix contracts')
    for column in BBO:
        d[column] = pd.to_numeric(d[column], errors='coerce')
    action = d.get('action', pd.Series('A', index=d.index)).map(lambda x: x.decode() if isinstance(x, bytes) else str(getattr(x, 'value', x)))
    n_trade = int(action.eq('T').sum())
    d['_reset_action'] = action.eq('R')
    if (d.ts_event.dropna().diff() < pd.Timedelta(0)).any():
        raise ValueError('ts_event decreases in arrival order; validate upstream ordering before detection')
    d = d.loc[~action.eq('T')].copy()
    if not d.empty:
        reset_in_group = d.groupby(['instrument_id', 'ts_event'], dropna=False)['_reset_action'].transform('any')
        d['_reset_action'] = reset_in_group
    duplicates = d.ts_event.notna() & d.duplicated(['instrument_id', 'ts_event'], keep='last')
    n_duplicates = int(duplicates.sum())
    d = d.loc[~duplicates].reset_index(drop=True)
    flags = pd.to_numeric(d.get('flags', pd.Series(0, index=d.index)), errors='coerce')
    flags_valid = flags.notna() & np.isfinite(flags) & flags.ge(0) & flags.le(255) & flags.mod(1).eq(0)
    safe_flags = flags.where(flags_valid, 0).astype('int64')
    on_grid = pd.Series(True, index=d.index)
    for column in ['bid_px_00', 'ask_px_00']:
        ticks = d[column] / settings.tick_size
        on_grid &= np.isfinite(ticks) & (ticks - ticks.round()).abs().le(1e-07)
    positive_sizes = np.isfinite(d.bid_sz_00) & np.isfinite(d.ask_sz_00) & d.bid_sz_00.gt(0) & d.ask_sz_00.gt(0)
    valid = d.ts_event.notna() & flags_valid & (safe_flags & 40).eq(0) & ~d._reset_action & on_grid & positive_sizes & d.bid_px_00.gt(0) & d.ask_px_00.lt(100000000.0) & d.ask_px_00.gt(d.bid_px_00)
    d['valid'] = valid
    d['midpoint'] = (d.bid_px_00 + d.ask_px_00) / 2
    d['price'] = d[{'midpoint': 'midpoint', 'bid': 'bid_px_00', 'ask': 'ask_px_00'}[settings.price_basis]]
    gaps = d.ts_event.diff() > pd.Timedelta(settings.max_quote_gap_ns, unit='ns')
    starts = valid & (~valid.shift(fill_value=False) | gaps)
    d['segment_id'] = starts.cumsum().astype('int64')
    d['comparable'] = valid & valid.shift(fill_value=False) & ~gaps
    d['price_before'] = d.price.shift()
    d['bid_before'] = d.bid_px_00.shift()
    d['ask_before'] = d.ask_px_00.shift()
    d['mid_before'] = d.midpoint.shift()
    diagnostics = dict(input_records=len(frame), trade_records_ignored=n_trade, same_timestamp_records_collapsed=n_duplicates, quote_states=len(d), invalid_states=int((~valid).sum()), missing_event_timestamps=int(d.ts_event.isna().sum()), continuity_segments=int(starts.sum()), price_basis=settings.price_basis)
    return (d, diagnostics)

def detect_events(frame: pd.DataFrame, settings: EventConfig=EventConfig()) -> EventResult:
    """Return quote states, all BBO price-change candidates and selected clusters."""
    quotes, diagnostics = prepare_quotes(frame, settings)
    changed = quotes.bid_px_00.ne(quotes.bid_before) | quotes.ask_px_00.ne(quotes.ask_before)
    q = quotes.loc[quotes.comparable & changed].copy()
    candidates = pd.DataFrame({'candidate_id': np.arange(len(q)), 'instrument_id': q.instrument_id.to_numpy(), 'segment_id': q.segment_id.to_numpy(), 't0': q.ts_event.to_numpy(), 'price_before': q.price_before.to_numpy(), 'price_after': q.price.to_numpy(), 'bid_before': q.bid_before.to_numpy(), 'bid_after': q.bid_px_00.to_numpy(), 'ask_before': q.ask_before.to_numpy(), 'ask_after': q.ask_px_00.to_numpy(), 'mid_before': q.mid_before.to_numpy(), 'mid_after': q.midpoint.to_numpy()})
    for prefix in ['bid', 'ask']:
        candidates[f'{prefix}_move_ticks'] = (candidates[f'{prefix}_after'] - candidates[f'{prefix}_before']) / settings.tick_size
    candidates['move_points'] = candidates.price_after - candidates.price_before
    candidates['move_ticks'] = candidates.move_points / settings.tick_size
    candidates['direction'] = np.select([candidates.move_ticks > 0, candidates.move_ticks < 0], ['up', 'down'], default='flat')
    candidates['size_class'] = _labels(candidates.move_ticks)
    candidates['is_price_move'] = candidates.move_ticks.ne(0)
    candidates['cluster_id'] = pd.Series(index=candidates.index, dtype='int64')
    cluster_frame = _cluster_candidates(candidates, quotes, settings)
    candidates = candidates.reindex(columns=CANDIDATE_COLUMNS)
    candidates['cluster_id'] = candidates.cluster_id.astype('int64')
    selected = cluster_frame.loc[cluster_frame.selected.astype(bool)].copy()
    selected.insert(0, 'event_id', selected.cluster_id)
    diagnostics.update(settings=asdict(settings), bbo_price_change_candidates=len(candidates), directional_candidates=int(candidates.is_price_move.sum()), clusters=len(cluster_frame), selected_events=len(selected), round_trip_clusters=int(cluster_frame.is_round_trip.sum()), reversal_clusters=int(cluster_frame.has_reversal.sum()), excluded_clusters=int((~cluster_frame.selected.astype(bool)).sum()))
    for table in (candidates, cluster_frame, selected):
        for column in ('t0', 'cluster_start', 'cluster_end', 'confirmed_at', 'observation_end'):
            if column in table:
                ts = pd.to_datetime(table[column], utc=True).astype('datetime64[ns, UTC]')
                values = ts.astype('int64').astype('Int64')
                table[column + '_ns'] = values.mask(ts.isna())
        if 'cluster_end_ns' in table:
            table['cluster_duration_ns'] = table.cluster_end_ns - table.cluster_start_ns
            table['confirmation_delay_ns'] = table.confirmed_at_ns - table.t0_ns
            table['observation_duration_ns'] = table.observation_end_ns - table.t0_ns
    return EventResult(quotes, candidates, cluster_frame, selected.reset_index(drop=True), diagnostics)


def _cluster_candidates(candidates, quotes, settings):
    """Build bounded clusters using integer ns and array reductions."""
    if candidates.empty:
        candidates['cluster_id'] = pd.Series(dtype='int64')
        return pd.DataFrame(columns=CLUSTER_COLUMNS)
    t = candidates.t0.astype('int64').to_numpy()
    seg = candidates.segment_id.to_numpy()
    start = np.zeros(len(t), dtype=bool)
    start[0] = True
    anchor = 0
    for j in range(1, len(t)):
        if (seg[j] != seg[j-1] or t[j]-t[j-1] > settings.cluster_gap_ns
                or t[j]-t[anchor] > settings.max_cluster_span_ns):
            start[j] = True
            anchor = j
    a = np.flatnonzero(start)
    b = np.r_[a[1:], len(t)]
    cid = np.cumsum(start)-1
    candidates['cluster_id'] = cid
    m = candidates.move_ticks.to_numpy()
    directional = m != 0
    count = np.add.reduceat(directional.astype('int64'), a)
    positive = np.add.reduceat((m > 0).astype('int64'), a)
    negative = np.add.reduceat((m < 0).astype('int64'), a)
    before = candidates.price_before.to_numpy()[a]
    after = candidates.price_after.to_numpy()[b-1]
    net = after-before
    ticks = net/settings.tick_size
    first, last = t[a], t[b-1]
    ids = seg[a]
    bounds = quotes.loc[quotes.valid].groupby('segment_id').ts_event.agg(['min','max'])
    left_bound = bounds['min'].astype('int64').reindex(ids).to_numpy()
    right_bound = bounds['max'].astype('int64').reindex(ids).to_numpy()
    left_linked = np.zeros(len(a), dtype=bool)
    right_linked = np.zeros(len(a), dtype=bool)
    if len(a)>1:
        linked = (ids[1:]==ids[:-1]) & (first[1:]-last[:-1]<=settings.cluster_gap_ns)
        left_linked[1:] = linked
        right_linked[:-1] = linked
    # Search within each continuity segment; a later segment is never a successor.
    sentinel = np.iinfo('int64').max
    next_move = np.full(len(a), sentinel, dtype='int64')
    dt = t[directional]
    ds = seg[directional]
    pos = np.searchsorted(dt, last, side='right')
    valid_pos = np.flatnonzero(pos<len(dt))
    valid_pos = valid_pos[ds[pos[valid_pos]]==ids[valid_pos]]
    next_move[valid_pos] = dt[pos[valid_pos]]
    desired = last+settings.reaction_window_ns
    limits = np.column_stack([desired,right_bound,next_move])
    which = limits.argmin(axis=1)
    end = limits[np.arange(len(a)),which]
    overlap = next_move<desired
    reversal = (positive>0)&(negative>0)
    round_trip = (count>0)&(ticks==0)
    left_missing = first-left_bound<settings.cluster_gap_ns
    right_missing = right_bound<last+settings.cluster_gap_ns
    reasons = np.full(len(a),'',dtype=object)
    def add(mask,label):
        idx=np.flatnonzero(mask)
        reasons[idx]=[str(reasons[i])+(';' if reasons[i] else '')+label for i in idx]
    add(count==0,'no_price_change')
    add(round_trip,'round_trip')
    if settings.drop_reversals:add(reversal,'reversal')
    add(left_linked|right_linked,'span_cap_split')
    add(left_missing,'missing_left_context')
    add(right_missing,'missing_right_context')
    if settings.singleton_only:add(count!=1,'not_singleton')
    if settings.overlap_policy=='drop':
        add(overlap,'overlap')
        add(right_bound<desired,'incomplete_horizon')
    add(end<=last,'empty_observation_window')
    def utc(x):return pd.to_datetime(x,utc=True,unit='ns')
    # Avoid float conversion of epoch ns when inserting NaT.
    next_ts=pd.to_datetime(np.where(next_move==sentinel,np.iinfo('int64').min,next_move),utc=True,unit='ns')
    return pd.DataFrame(dict(cluster_id=np.arange(len(a)),
        instrument_id=candidates.instrument_id.to_numpy()[a],segment_id=ids,
        cluster_start=utc(first),cluster_end=utc(last),t0=utc(last),
        confirmed_at=utc(last+settings.cluster_gap_ns),price_before=before,
        price_after=after,move_points=net,move_ticks=ticks,abs_move_ticks=np.abs(ticks),
        direction=np.select([ticks>0,ticks<0],['up','down'],default='flat'),
        size_class=_labels(ticks),n_updates=b-a,n_price_moves=count,
        gross_move_ticks=np.add.reduceat(np.abs(m),a),has_reversal=reversal,
        is_round_trip=round_trip,left_linked=left_linked,right_linked=right_linked,
        left_context_missing=left_missing,right_context_missing=right_missing,
        next_price_move=next_ts,segment_end=utc(right_bound),observation_end=utc(end),
        observation_end_reason=np.array(['horizon','segment_end','next_price_move'])[which],
        overlap_within_horizon=overlap,horizon_complete=end==desired,
        selected=reasons=='',exclusion_reason=reasons),columns=CLUSTER_COLUMNS)


## Filtering and timestamp assignment

Adjacent BBO price candidates join a cluster only if both conditions hold:

- adjacent gap <= `cluster_gap_ns` (default 250,000 ns);
- elapsed time from the first member <= `max_cluster_span_ns` (2,000,000 ns).

A zero-net round trip is excluded. Partial reversals are excluded by default.
Bursts split only by the span cap are flagged and excluded rather than counted
as several independent moves. Missing quiet context at either boundary is
reported. Invalid states, resets and gaps are never bridged.

Two outputs answer different questions:

| Table | Move and timestamp | Intended use |
| --- | --- | --- |
| `events` | One primary-price update; `t0` is that update's `ts_event` | Default trigger table for an instantaneous-event specification. Native multi-tick jumps are included. |
| `clustered_events` | Net change over a bounded path; `t0 = cluster_end` | Sensitivity / path analysis, not an instantaneous jump at the first member. |

`confirmed_at = cluster_end + cluster_gap_ns` records when quiet-gap selection
could first be confirmed. Filtering is retrospective: even single-update
selection uses subsequent observations. It is not a tradable online signal.
The event's time is not moved to confirmation time, and an aggregate's full
net move is never backdated to its first member.

Observation windows are `[t0, observation_end)`. They end at the earliest of
the configured horizon, the next raw primary-price move (even if later filtered
out), or the last valid state in that continuity segment. Truncation is recorded,
not interpreted as option resolution. `overlap_policy='drop'` instead requires
the full horizon without a subsequent move.


In [5]:
def single_update_events(result: EventResult, settings: EventConfig) -> pd.DataFrame:
    """Return isolated single midpoint changes at their actual exchange time."""
    table=result.events.loc[result.events.n_price_moves.eq(1)].copy()
    if table.empty:
        table['event_kind']='single_price_update'
        return table.reset_index(drop=True)
    raw=result.candidates.loc[result.candidates.is_price_move].copy()
    raw=raw.loc[raw.cluster_id.isin(table.cluster_id)].set_index('cluster_id')
    members=raw.reindex(table.cluster_id)
    t=members.t0.astype('int64').to_numpy()
    desired=t+settings.reaction_window_ns
    next_ts=table.next_price_move.astype('int64').to_numpy()
    next_ns=np.where(table.next_price_move.isna(),np.iinfo('int64').max,next_ts)
    limits=np.column_stack([desired,table.segment_end.astype('int64').to_numpy(),next_ns])
    which=limits.argmin(axis=1)
    end=limits[np.arange(len(t)),which]
    table['t0']=pd.to_datetime(t,utc=True,unit='ns')
    table['t0_ns']=pd.array(t,dtype='Int64')
    table['price_before']=members.price_before.to_numpy()
    table['price_after']=members.price_after.to_numpy()
    table['observation_end']=pd.to_datetime(end,utc=True,unit='ns')
    table['observation_end_ns']=pd.array(end,dtype='Int64')
    table['observation_end_reason']=np.array(['horizon','segment_end','next_price_move'])[which]
    table['overlap_within_horizon']=next_ns<desired
    table['horizon_complete']=end==desired
    table['confirmation_delay_ns']=pd.array(table.confirmed_at.astype('int64').to_numpy()-t,dtype='Int64')
    table['observation_duration_ns']=pd.array(end-t,dtype='Int64')
    keep=end>t
    if settings.overlap_policy=='drop':keep &= (end==desired)&(next_ns>=desired)
    table=table.loc[keep].copy()
    table['event_kind']='single_price_update'
    return table.reset_index(drop=True)


## Local data and offline example

The loader validates Issue #3's `futures.json`: dataset, schema, date, instrument
ID, outright ES symbol, and request interval. It uses the selected underlying
from that manifest, including during a roll window. No lead contract is chosen
again. DBN prices are explicitly decoded as floating index points.

Only futures are loaded. The selected date's BBO columns must fit in memory.
The demo's instrument ID and prices are synthetic and are labeled as such.


In [6]:
"""Read Issue #3's local futures cache and write Issue #4 outputs."""
from dataclasses import dataclass
from datetime import date
import inspect
import json
from pathlib import Path
import re
import pandas as pd

@dataclass
class FuturesSample:
    quotes: pd.DataFrame
    manifest: dict
    path: Path

def load_futures_sample(trading_date: date, raw_dir: Path=config.RAW_DIR) -> FuturesSample:
    """Load the manifest-selected underlying, never a recomputed lead future.

    Only MBP-1 futures are decoded; options and MBO files are not read. Chunked
    conversion reduces peak memory from unused DBN fields, but the selected
    futures BBO columns for this date must still fit in memory.
    """
    folder = Path(raw_dir) / 'mbp-1' / f'{trading_date:%Y%m%d}'
    path, manifest_path = (folder / 'futures.dbn.zst', folder / 'futures.json')
    if not path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f'Missing {path} or its futures.json manifest. Run scripts/download.py --dates {trading_date} first.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    for name, expected in {'dataset': config.DATASET, 'schema': 'mbp-1', 'kind': 'futures', 'trading_date': trading_date.isoformat(), 'stype_in': 'instrument_id'}.items():
        if manifest.get(name) != expected:
            raise ValueError(f'Invalid futures manifest {name}: expected {expected!r}')
    ids, symbols = (manifest.get('symbols'), manifest.get('raw_symbols'))
    if not isinstance(ids, list) or len(ids) != 1 or (not isinstance(symbols, list)) or (len(symbols) != 1):
        raise ValueError('The manifest must identify exactly one futures instrument and symbol')
    if not re.fullmatch('ES[HMUZ]\\d{1,2}', str(symbols[0])):
        raise ValueError('Manifest symbol is not an outright ES future')
    try:
        expected_id = int(ids[0])
    except (ValueError, TypeError):
        raise ValueError('Manifest instrument ID is not an integer') from None
    if str(expected_id) != str(ids[0]) or expected_id < 0:
        raise ValueError('Manifest instrument ID is not a nonnegative integer')
    start = pd.to_datetime(manifest.get('start'), utc=True, errors='coerce')
    end = pd.to_datetime(manifest.get('end'), utc=True, errors='coerce')
    if pd.isna(start) or pd.isna(end) or start >= end:
        raise ValueError('Manifest needs a valid start/end interval')
    import databento as db
    store = db.DBNStore.from_file(path)
    dbn_schema = getattr(getattr(store, 'metadata', None), 'schema', None)
    if dbn_schema is not None and str(dbn_schema) != 'mbp-1':
        raise ValueError('DBN metadata schema is not mbp-1')
    signature = inspect.signature(store.to_df).parameters
    kwargs = {'pretty_ts': True, 'map_symbols': False}
    if 'price_type' in signature:
        kwargs['price_type'] = 'float'
    else:
        kwargs['pretty_px'] = True
    if 'count' in signature:
        kwargs['count'] = 500000
    converted = store.to_df(**kwargs)
    chunks = [converted] if isinstance(converted, pd.DataFrame) else converted
    columns = ['ts_event', 'ts_recv', 'instrument_id', 'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'flags', 'action', 'sequence']
    frames = []
    for chunk in chunks:
        if chunk.index.name == 'ts_recv' and 'ts_recv' not in chunk:
            chunk = chunk.reset_index()
        if 'instrument_id' not in chunk:
            raise ValueError('DBN data is missing instrument_id')
        if not pd.to_numeric(chunk.instrument_id, errors='coerce').eq(expected_id).all():
            raise ValueError('DBN instrument IDs do not match the selected underlying in futures.json')
        if 'ts_recv' in chunk:
            recv = pd.to_datetime(chunk.ts_recv, utc=True, errors='coerce')
            if (recv.notna() & ((recv < start) | (recv >= end))).any():
                raise ValueError('DBN reception timestamps fall outside the manifest request interval')
        frames.append(chunk.loc[:, [c for c in columns if c in chunk]].copy())
    quotes = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    if quotes.empty:
        raise ValueError('Cached futures file has no records')
    return FuturesSample(quotes, manifest, path)

def write_event_result(result: EventResult, folder: Path, source: dict) -> dict[str, Path]:
    """Write audit tables, selected events and diagnostics to the chosen folder."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    outputs = {}
    for name, frame in {'candidates': result.candidates, 'clusters': result.clusters, 'events': result.events}.items():
        target = folder / f'{name}.parquet'
        partial = target.with_suffix('.partial.parquet')
        frame.to_parquet(partial, index=False)
        partial.replace(target)
        outputs[name] = target
    target = folder / 'diagnostics.json'
    target.write_text(json.dumps({**result.diagnostics, 'source': source}, indent=2, default=str), encoding='utf-8')
    outputs['diagnostics'] = target
    return outputs


In [7]:
"""Small offline example and optional command-line wrapper."""
import argparse
from datetime import date
from pathlib import Path
import pandas as pd

def demo_quotes() -> pd.DataFrame:
    """Small deterministic quote path for an offline smoke test, not market data."""
    base = pd.Timestamp('2026-10-07T13:30:00Z')
    return pd.DataFrame({'ts_event': [base + pd.Timedelta(t * 1000, unit='ns') for t in [0, 1000, 1100, 20000, 20100, 40000, 60000]], 'instrument_id': 1, 'bid_px_00': [7000, 7000.25, 7000.5, 7000.75, 7000.5, 7000.25, 7000.25], 'ask_px_00': [7000.25, 7000.5, 7000.75, 7001, 7000.75, 7000.5, 7000.5], 'bid_sz_00': 10, 'ask_sz_00': 10, 'action': 'A', 'flags': 0})

def main(argv: list[str] | None=None) -> None:
    parser = argparse.ArgumentParser(description='Detect ES futures moves from local Issue #3 MBP-1 files; no API calls.')
    parser.add_argument('--dates', nargs='+', type=date.fromisoformat, default=config.STUDY_DATES)
    parser.add_argument('--raw-dir', type=Path, default=config.RAW_DIR)
    parser.add_argument('--out-dir', type=Path, default=OUT_DIR)
    parser.add_argument('--basis', choices=['midpoint', 'bid', 'ask'], default='midpoint')
    parser.add_argument('--cluster-gap-ns', type=int, default=config.CLUSTER_GAP_NS)
    parser.add_argument('--max-cluster-span-ns', type=int, default=config.MAX_CLUSTER_SPAN_NS)
    parser.add_argument('--reaction-window-ns', type=int, default=config.REACTION_WINDOW_NS)
    parser.add_argument('--max-quote-gap-ns', type=int, default=config.MAX_QUOTE_GAP_NS)
    parser.add_argument('--keep-reversals', action='store_true', help='retain partial reversals; zero-net clusters remain excluded')
    parser.add_argument('--singleton-only', action='store_true', help='retain clusters with exactly one primary-price move')
    parser.add_argument('--overlap-policy', choices=['truncate', 'drop'], default='truncate')
    parser.add_argument('--demo', action='store_true', help='run synthetic offline example instead of reading market data')
    args = parser.parse_args(argv)
    try:
        settings = EventConfig(price_basis=args.basis, cluster_gap_ns=args.cluster_gap_ns, max_cluster_span_ns=args.max_cluster_span_ns, reaction_window_ns=args.reaction_window_ns, max_quote_gap_ns=args.max_quote_gap_ns, drop_reversals=not args.keep_reversals, singleton_only=args.singleton_only, overlap_policy=args.overlap_policy)
        if args.demo:
            result = detect_events(demo_quotes(), settings)
            paths = write_event_result(result, args.out_dir / 'demo', {'kind': 'synthetic_demo'})
            print(f'SYNTHETIC DEMO: {len(result.candidates)} candidates -> {len(result.events)} selected events')
            print(result.events[['t0', 'price_before', 'price_after', 'direction', 'move_points', 'move_ticks']].to_string(index=False))
            print(f"outputs: {paths['events'].parent}")
            return
        for day in dict.fromkeys(args.dates):
            sample = load_futures_sample(day, args.raw_dir)
            result = detect_events(sample.quotes, settings)
            source = {'kind': 'market_data', 'path': str(sample.path), 'manifest': sample.manifest}
            paths = write_event_result(result, args.out_dir / f'{day:%Y%m%d}', source)
            print(f"{day}: {sample.manifest['raw_symbols'][0]}: {len(result.candidates):,} candidates -> {len(result.events):,} selected events")
            print(f"outputs: {paths['events'].parent}")
    except (ValueError, FileNotFoundError) as exc:
        parser.exit(2, f'event detection: {exc}\n')


## Offline tests

Run the detector and IO regression suite in a temporary directory. It includes
same-timestamp price/size updates, invalid states, reversals, half ticks,
bounded clusters, overlap, ns boundaries, timestamp assignment, and a native
compressed DBN-to-Parquet round trip. The tests do not use a network or API key.


In [8]:
# Publish the in-memory functions for pytest; no source files are installed in the repo.
import pytest
engine_module=types.ModuleType('issue4_notebook_engine')
engine_module.__dict__.update({k:v for k,v in list(globals().items()) if not k.startswith('_')})
sys.modules['issue4_notebook_engine']=engine_module
TEST_SOURCE = 'from issue4_notebook_engine import *\n"""Offline tests for quote states, tick classification and event windows."""\nfrom dataclasses import replace\nimport numpy as np\nimport pandas as pd\nimport pytest\nBASE = pd.Timestamp(\'2026-10-07T13:30:00Z\')\nCFG = EventConfig(cluster_gap_ns=100000, max_cluster_span_ns=1000000, reaction_window_ns=5000000)\n\ndef quotes(bids, times=None, **extra):\n    if times is None:\n        times = [0, *range(1000, 1000 * (len(bids) - 1), 1000), 20000] if len(bids) > 1 else [0]\n    d = pd.DataFrame({\'ts_event\': [BASE + pd.Timedelta(t * 1000, unit=\'ns\') for t in times], \'instrument_id\': 10252, \'bid_px_00\': np.array(bids, dtype=float), \'ask_px_00\': np.array(bids, dtype=float) + 0.25, \'bid_sz_00\': 10.0, \'ask_sz_00\': 10.0, \'action\': \'A\', \'flags\': 0})\n    for k, v in extra.items():\n        d[k] = v\n    return d\n\ndef test_single_tick_and_required_fields():\n    r = detect_events(quotes([7000, 7000.25, 7000.25]), CFG)\n    e = r.events.iloc[0]\n    assert e.t0 == BASE + pd.Timedelta(1000000, unit=\'ns\')\n    assert e.price_before == 7000.125 and e.price_after == 7000.375\n    assert e.direction == \'up\' and e.move_points == 0.25 and (e.move_ticks == 1)\n    assert e.size_class == \'single_tick\' and e.horizon_complete\n\n@pytest.mark.parametrize(\'move,direction,size\', [(0.5, \'up\', \'multi_tick\'), (-0.75, \'down\', \'multi_tick\'), (-0.25, \'down\', \'single_tick\')])\ndef test_move_size(move, direction, size):\n    e = detect_events(quotes([7000, 7000 + move, 7000 + move]), CFG).events.iloc[0]\n    assert e.direction == direction and e.size_class == size and (e.move_ticks == move / 0.25)\n\ndef test_half_tick_midpoint_not_rounded():\n    d = quotes([7000] * 3)\n    d.loc[1:, \'ask_px_00\'] = 7000.5\n    e = detect_events(d, CFG).events.iloc[0]\n    assert e.move_points == 0.125 and e.move_ticks == 0.5 and (e.size_class == \'sub_tick\')\n\ndef test_same_timestamp_size_update_does_not_erase_price_move():\n    d = quotes([7000, 7000.25, 7000.25, 7000.25], [0, 1000, 1000, 20000])\n    d.loc[2, \'bid_sz_00\'] = 20\n    r = detect_events(d, CFG)\n    assert r.events.move_ticks.tolist() == [1]\n    assert r.diagnostics[\'same_timestamp_records_collapsed\'] == 1\n\ndef test_same_timestamp_round_trip_has_no_final_price_change():\n    r = detect_events(quotes([7000, 7000.25, 7000, 7000], [0, 1000, 1000, 20000]), CFG)\n    assert r.candidates.empty and r.events.empty\n\ndef test_size_only_updates_are_not_candidates():\n    assert detect_events(quotes([7000] * 3, bid_sz_00=[10, 20, 30]), CFG).candidates.empty\n\ndef test_trade_prints_do_not_change_quote_state():\n    d = quotes([7000, 7100, 7000], [0, 1000, 20000], action=[\'A\', \'T\', \'A\'])\n    r = detect_events(d, CFG)\n    assert r.candidates.empty and r.diagnostics[\'trade_records_ignored\'] == 1\n\ndef test_trade_after_quote_at_same_time_cannot_roll_back_quote():\n    d = quotes([7000, 7000.25, 7000, 7000.25], [0, 1000, 1000, 20000], action=[\'A\', \'A\', \'T\', \'A\'])\n    assert detect_events(d, CFG).events.move_ticks.tolist() == [1]\n\n@pytest.mark.parametrize(\'column,value\', [(\'bid_sz_00\', 0), (\'bid_px_00\', np.nan), (\'ask_px_00\', 6999), (\'ask_px_00\', 7000), (\'flags\', 8), (\'flags\', 32), (\'flags\', np.nan), (\'bid_px_00\', 7000.01), (\'ask_px_00\', 9220000000.0), (\'bid_sz_00\', np.inf), (\'ts_event\', pd.NaT), (\'action\', \'R\')])\ndef test_invalid_state_resets_comparison(column, value):\n    d = quotes([7000, 7000.25, 7000.5, 7000.5], [0, 1000, 2000, 20000])\n    d.loc[1, column] = value\n    assert detect_events(d, CFG).candidates.empty\n\ndef test_final_invalid_same_timestamp_resets():\n    d = quotes([7000, 7000.25, 7000.25, 7000.5, 7000.5], [0, 1000, 1000, 2000, 20000])\n    d.loc[2, \'bid_sz_00\'] = 0\n    assert detect_events(d, CFG).events.empty\n\ndef test_large_data_gap_reinitializes():\n    d = quotes([7000, 7001, 7001], [0, 2000000, 2020000])\n    assert detect_events(d, CFG).candidates.empty\n\ndef test_missing_required_fields_raise():\n    with pytest.raises(ValueError, match=\'Missing columns\'):\n        detect_events(quotes([7000]).drop(columns=\'ask_sz_00\'))\n\ndef test_reordered_input_is_not_silently_sorted():\n    with pytest.raises(ValueError, match=\'decreases\'):\n        detect_events(quotes([7000, 7000.25, 7000.5], [0, 2000, 1000]))\n\ndef test_multiple_futures_rejected():\n    d = quotes([7000] * 3)\n    d.loc[1, \'instrument_id\'] = 99\n    with pytest.raises(ValueError, match=\'one selected\'):\n        detect_events(d)\n\ndef test_cluster_t0_is_final_update_not_start():\n    r = detect_events(quotes([7000, 7000.25, 7000.5, 7000.5], [0, 1000, 1050, 20000]), CFG)\n    e = r.events.iloc[0]\n    assert len(r.candidates) == 2 and len(r.events) == 1\n    assert e.cluster_start == BASE + pd.Timedelta(1000000, unit=\'ns\')\n    assert e.t0 == BASE + pd.Timedelta(1050000, unit=\'ns\') and e.move_ticks == 2\n    assert e.confirmed_at == BASE + pd.Timedelta(1150000, unit=\'ns\')\n    assert r.candidates.cluster_id.nunique() == 1\n\ndef test_round_trip_suppressed_but_audited():\n    r = detect_events(quotes([7000, 7000.25, 7000, 7000], [0, 1000, 1050, 20000]), CFG)\n    assert r.events.empty and r.clusters.is_round_trip.iloc[0]\n    assert \'round_trip\' in r.clusters.exclusion_reason.iloc[0]\n\ndef test_partial_reversal_filter_can_be_disabled():\n    d = quotes([7000, 7000.5, 7000.25, 7000.25], [0, 1000, 1050, 20000])\n    assert detect_events(d, CFG).events.empty\n    e = detect_events(d, replace(CFG, drop_reversals=False)).events.iloc[0]\n    assert e.move_ticks == 1 and e.gross_move_ticks == 3 and e.has_reversal\n\ndef test_span_cap_does_not_make_artificial_independent_events():\n    d = quotes([7000, 7000.25, 7000.5, 7000.75, 7001, 7001], [0, 1000, 1090, 1180, 1270, 20000])\n    r = detect_events(d, replace(CFG, max_cluster_span_ns=150000))\n    assert len(r.clusters) == 2 and r.events.empty\n    assert r.clusters.exclusion_reason.str.contains(\'span_cap_split\').all()\n\ndef test_singleton_option_excludes_aggregates():\n    d = quotes([7000, 7000.25, 7000.5, 7000.5], [0, 1000, 1050, 20000])\n    assert detect_events(d, replace(CFG, singleton_only=True)).events.empty\n\ndef test_observation_window_stops_at_next_raw_move_even_if_suppressed():\n    d = quotes([7000, 7000.25, 7000.5, 7000.25, 7000.25], [0, 1000, 3000, 3050, 20000])\n    r = detect_events(d, CFG)\n    assert len(r.events) == 1\n    e = r.events.iloc[0]\n    assert e.observation_end == BASE + pd.Timedelta(3000000, unit=\'ns\')\n    assert e.observation_end_reason == \'next_price_move\' and e.overlap_within_horizon\n    assert not e.horizon_complete\n\ndef test_drop_overlap_policy():\n    d = quotes([7000, 7000.25, 7000.5, 7000.5], [0, 1000, 3000, 20000])\n    r = detect_events(d, replace(CFG, overlap_policy=\'drop\'))\n    assert r.events.move_ticks.tolist() == [1] and r.events.t0.iloc[0] == BASE + pd.Timedelta(3000000, unit=\'ns\')\n    assert \'overlap\' in r.clusters.exclusion_reason.iloc[0]\n\ndef test_windows_are_non_overlapping():\n    d = quotes([7000, 7000.25, 7000.5, 7000.75, 7000.75], [0, 1000, 3000, 5000, 20000])\n    e = detect_events(d, CFG).events\n    assert len(e) == 3\n    assert (e.observation_end.iloc[:-1].reset_index(drop=True) <= e.t0.iloc[1:].reset_index(drop=True)).all()\n\ndef test_unconfirmed_tail_excluded():\n    r = detect_events(quotes([7000, 7000.25], [0, 1000]), CFG)\n    assert r.events.empty and r.clusters.right_context_missing.iloc[0]\n\ndef test_missing_left_context_excluded():\n    r = detect_events(quotes([7000, 7000.25, 7000.25], [0, 50, 20000]), CFG)\n    assert r.events.empty and r.clusters.left_context_missing.iloc[0]\n\ndef test_truncated_sample_horizon_is_disclosed():\n    e = detect_events(quotes([7000, 7000.25, 7000.25], [0, 1000, 2000]), CFG).events.iloc[0]\n    assert e.observation_end_reason == \'segment_end\' and (not e.horizon_complete)\n\ndef test_spread_change_without_midpoint_move_is_audited():\n    d = quotes([7000] * 3)\n    d.loc[1:, \'bid_px_00\'] = 6999.75\n    d.loc[1:, \'ask_px_00\'] = 7000.5\n    r = detect_events(d, CFG)\n    assert len(r.candidates) == 1 and r.events.empty\n    assert r.candidates.direction.iloc[0] == \'flat\'\n    e = detect_events(d, replace(CFG, price_basis=\'ask\')).events.iloc[0]\n    assert e.move_ticks == 1 and e.direction == \'up\'\n\ndef test_empty_and_no_moves_have_stable_columns():\n    for d in [quotes([7000]).iloc[:0], quotes([7000] * 3)]:\n        r = detect_events(d, CFG)\n        assert r.events.empty and r.candidates.empty\n        assert {\'t0\', \'price_before\', \'price_after\', \'move_ticks\', \'direction\'} <= set(r.events)\n\n@pytest.mark.parametrize(\'kw\', [{\'cluster_gap_ns\': -1}, {\'tick_size\': 0}, {\'reaction_window_ns\': float(\'nan\')}, {\'max_cluster_span_ns\': 50, \'cluster_gap_ns\': 100}, {\'price_basis\': \'trade\'}, {\'overlap_policy\': \'unknown\'}])\ndef test_bad_configuration(kw):\n    with pytest.raises(ValueError):\n        EventConfig(**kw)\n\ndef test_nanosecond_separation_preserved():\n    d = quotes([7000, 7000.25, 7000.5, 7000.5], [0, 1000, 1001, 20000])\n    d.loc[2, \'ts_event\'] = d.loc[1, \'ts_event\'] + pd.Timedelta(123, unit=\'ns\')\n    r = detect_events(d, replace(CFG, cluster_gap_ns=0))\n    assert r.events.t0.iloc[1].value - r.events.t0.iloc[0].value == 123\n\ndef test_clear_followed_by_quote_at_same_timestamp_resets():\n    d = quotes([7000, 7000, 7001, 7001], [0, 1000, 1000, 20000], action=[\'A\', \'R\', \'A\', \'A\'])\n    assert detect_events(d, CFG).events.empty\n\'Manifest integration and a native compressed DBN round trip, without an API.\'\nfrom datetime import date\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport pandas as pd\nimport pytest\nDAY = date(2026, 10, 7)\n\n@pytest.fixture\ndef native_sample(tmp_path):\n    import databento as db\n    import databento_dbn as dbn\n    folder = tmp_path / \'raw\' / \'mbp-1\' / \'20261007\'\n    folder.mkdir(parents=True)\n    manifest = dict(trading_date=\'2026-10-07\', kind=\'futures\', dataset=\'GLBX.MDP3\', schema=\'mbp-1\', stype_in=\'instrument_id\', symbols=[10252], raw_symbols=[\'ESZ6\'], start=BASE.isoformat(), end=(BASE + pd.Timedelta(1, unit=\'s\')).isoformat())\n    (folder / \'futures.json\').write_text(json.dumps(manifest), encoding=\'utf-8\')\n    metadata = dbn.Metadata(dataset=\'GLBX.MDP3\', start=BASE.value, stype_in=dbn.SType(\'instrument_id\'), stype_out=dbn.SType(\'instrument_id\'), schema=dbn.Schema(\'mbp-1\'))\n    records = []\n    for row in quotes([7000, 7000.25, 7000.25]).itertuples():\n        records.append(dbn.MBP1Msg(publisher_id=1, instrument_id=10252, ts_event=row.ts_event.value, ts_recv=row.ts_event.value + 10000, price=int(row.bid_px_00 * 1000000000.0), size=10, action=dbn.Action(\'A\'), side=dbn.Side(\'B\'), depth=0, levels=dbn.BidAskPair(bid_px=int(row.bid_px_00 * 1000000000.0), ask_px=int(row.ask_px_00 * 1000000000.0), bid_sz=10, ask_sz=10)))\n    uncompressed = tmp_path / \'synthetic.dbn\'\n    uncompressed.write_bytes(bytes(metadata) + b\'\'.join((bytes(r) for r in records)))\n    db.DBNStore.from_file(uncompressed).to_file(folder / \'futures.dbn.zst\', compression=\'zstd\')\n    return (tmp_path / \'raw\', folder, manifest)\n\ndef test_real_dbn_decoder_and_parquet_roundtrip(native_sample, tmp_path):\n    raw, folder, _ = native_sample\n    sample = load_futures_sample(DAY, raw)\n    assert sample.quotes.bid_px_00.iloc[0] == 7000\n    assert sample.manifest[\'raw_symbols\'] == [\'ESZ6\']\n    result = detect_events(sample.quotes, CFG)\n    assert result.events.move_ticks.tolist() == [1]\n    paths = write_event_result(result, tmp_path / \'out\', {\'kind\': \'synthetic_fixture\'})\n    recovered = pd.read_parquet(paths[\'events\'])\n    assert recovered.t0.iloc[0].value == result.events.t0.iloc[0].value\n    assert recovered.price_before.iloc[0] == 7000.125\n    assert json.loads(paths[\'diagnostics\'].read_text())[\'source\'][\'kind\'] == \'synthetic_fixture\'\n\ndef test_preserves_selected_future_during_roll(native_sample):\n    raw, folder, m = native_sample\n    m[\'raw_symbols\'] = [\'ESU6\']\n    (folder / \'futures.json\').write_text(json.dumps(m))\n    assert load_futures_sample(DAY, raw).manifest[\'raw_symbols\'] == [\'ESU6\']\n\n@pytest.mark.parametrize(\'field,value\', [(\'schema\', \'mbo\'), (\'kind\', \'options\'), (\'dataset\', \'OTHER\'), (\'trading_date\', \'2026-10-08\'), (\'stype_in\', \'raw_symbol\'), (\'symbols\', [999]), (\'symbols\', [10252, 99]), (\'raw_symbols\', [\'MESZ6\']), (\'start\', \'bad\'), (\'end\', \'2026-10-07T00:00:00Z\')])\ndef test_bad_manifest_rejected(native_sample, field, value):\n    raw, folder, m = native_sample\n    m[field] = value\n    (folder / \'futures.json\').write_text(json.dumps(m))\n    with pytest.raises(ValueError):\n        load_futures_sample(DAY, raw)\n\ndef test_missing_data_gives_download_instruction(tmp_path):\n    with pytest.raises(FileNotFoundError, match=\'scripts/download.py\'):\n        load_futures_sample(DAY, tmp_path)\n\ndef test_cli_real_file_path_uses_local_sample(native_sample, tmp_path, capsys):\n    raw, _, _ = native_sample\n    main([\'--dates\', \'2026-10-07\', \'--raw-dir\', str(raw), \'--out-dir\', str(tmp_path / \'events\')])\n    assert (tmp_path / \'events/20261007/events.parquet\').is_file()\n    assert \'ESZ6\' in capsys.readouterr().out\n\ndef test_cli_missing_data_is_error_not_fake_success(tmp_path, capsys):\n    with pytest.raises(SystemExit) as exc:\n        main([\'--dates\', \'2026-10-07\', \'--raw-dir\', str(tmp_path)])\n    assert exc.value.code == 2\n    assert \'Missing\' in capsys.readouterr().err\n\ndef test_demo_runs_without_api_key(tmp_path, monkeypatch, capsys):\n    monkeypatch.delenv(\'DATABENTO_API_KEY\', raising=False)\n    monkeypatch.delenv(\'DATABENTO_KEY\', raising=False)\n    main([\'--demo\', \'--out-dir\', str(tmp_path)])\n    events = pd.read_parquet(tmp_path / \'demo/events.parquet\')\n    assert events.move_ticks.tolist() == [2, -1]\n    assert \'SYNTHETIC DEMO\' in capsys.readouterr().out\n\ndef test_integer_nanosecond_parameters():\n    with pytest.raises(ValueError, match=\'integer\'):\n        EventConfig(cluster_gap_ns=0.5)\n    with pytest.raises(ValueError, match=\'integer\'):\n        EventConfig(reaction_window_ns=True)\n\n\ndef test_exact_nanosecond_boundary():\n    d=quotes([7000,7000.25,7000.5,7000.5],[0,1000,1001,20_000])\n    d.loc[2,\'ts_event\']=d.loc[1,\'ts_event\']+pd.Timedelta(123,unit=\'ns\')\n    r=detect_events(d,replace(CFG,cluster_gap_ns=123))\n    assert len(r.events)==1 and r.events.cluster_duration_ns.iloc[0]==123\n    assert r.events.t0_ns.iloc[0]==d.ts_event.iloc[2].value\n    r=detect_events(d,replace(CFG,cluster_gap_ns=122))\n    assert len(r.events)==2\n\n\ndef test_single_update_uses_actual_price_change_timestamp():\n    d=quotes([7000,7000,7000,7000],[0,1000,1050,20_000])\n    d.loc[1:,\'ask_px_00\']=7000.75\n    d.loc[2:,\'bid_px_00\']=6999.75\n    d.loc[2:,\'ask_px_00\']=7001.0  # spread changes, midpoint unchanged\n    r=detect_events(d,CFG)\n    s=single_update_events(r,CFG)\n    assert len(s)==1\n    assert s.t0.iloc[0]==BASE+pd.Timedelta(1_000_000,unit=\'ns\')\n    assert s.cluster_end.iloc[0]==BASE+pd.Timedelta(1_050_000,unit=\'ns\')\n    assert s.move_points.iloc[0]==.25\n\n\ndef test_selected_windows_remain_non_overlapping_in_ns():\n    r=detect_events(quotes([7000,7000.25,7000.5,7000.5],[0,1000,3000,20_000]),CFG)\n    s=single_update_events(r,CFG)\n    assert (s.observation_end_ns.iloc[:-1].reset_index(drop=True)<=s.t0_ns.iloc[1:].reset_index(drop=True)).all()\n'
class TestResults:
    def __init__(self):
        self.passed = self.failed = self.skipped = 0
    def pytest_runtest_logreport(self, report):
        if report.when == 'call':
            self.passed += int(report.passed)
            self.failed += int(report.failed)
            self.skipped += int(report.skipped)
TEST_RESULTS=TestResults()
with tempfile.TemporaryDirectory(prefix='issue4_tests_') as temp:
    path=Path(temp)/'test_issue4_engine.py'
    path.write_text(TEST_SOURCE,encoding='utf-8')
    TEST_EXIT_CODE=int(pytest.main(['-q',str(path),'--noconftest','--import-mode=importlib','-p','no:cacheprovider'],plugins=[TEST_RESULTS]))
if TEST_EXIT_CODE:
    raise RuntimeError('Offline tests failed; inspect the results before analyzing data.')
print('Offline regression suite passed.')


....................................................................     [100%]
68 passed in 3.24s
Offline regression suite passed.


## Run the sample

Read the manifest-selected future, check the quote states, and detect events.
All five dates must be present in real mode. Each input folder is
`data/raw/mbp-1/YYYYMMDD/` and contains `futures.dbn.zst` and `futures.json`.

Each day is processed separately, so events and clusters never cross dates.
We use the cached request interval without adding a new download-window
requirement. The detector is repeated at four clustering gaps on each day's
input. This is a filter comparison, not four separate datasets.


In [9]:
SETTINGS=EventConfig(cluster_gap_ns=CLUSTER_GAP_NS,max_cluster_span_ns=MAX_CLUSTER_SPAN_NS,
    reaction_window_ns=REACTION_WINDOW_NS,max_quote_gap_ns=MAX_QUOTE_GAP_NS)
available=[]; missing=[]
for day in dict.fromkeys(STUDY_DATES):
    folder=RAW_DIR/'mbp-1'/f'{day:%Y%m%d}'
    (available if (folder/'futures.dbn.zst').is_file() and (folder/'futures.json').is_file() else missing).append(day)
if RUN_MODE=='real' and missing:
    raise FileNotFoundError(f'Missing cached futures and manifest for {missing}. Download separately with Issue #3 first.')
USE_DEMO=RUN_MODE=='demo' or (RUN_MODE=='auto' and not available)
if missing and RUN_MODE=='auto':print('Requested dates not in the local cache:',missing)
RESULTS={}; SOURCES={}; RUN_SUMMARIES=[]; SENSITIVITY_ROWS=[]
jobs=['demo'] if USE_DEMO else available
for key in jobs:
    if key=='demo':
        frame=demo_quotes();source={'kind':'synthetic_demo','label':'No market-data validation'}
        print('SYNTHETIC DEMO ONLY: no requested cached market-data sample is being analyzed.')
    else:
        sample=load_futures_sample(key,RAW_DIR)
        frame=sample.quotes;source={'kind':'market_data','path':str(sample.path),'manifest':sample.manifest}
    print(f'Analyzing {key}: {len(frame):,} records',flush=True)
    result=detect_events(frame,SETTINGS)
    events=single_update_events(result,SETTINGS)
    RESULTS[str(key)]={'result':result,'events':events}
    SOURCES[str(key)]=source
    RUN_SUMMARIES.append({'sample':str(key),'source_kind':source['kind'],
        'records':len(frame),'candidates':len(result.candidates),
        'directional_candidates':result.diagnostics['directional_candidates'],
        'clusters':len(result.clusters),'clustered_events':len(result.events),
        'single_update_events':len(events),
        'multi_tick_single_updates':int(events.size_class.eq('multi_tick').sum()),
        'truncated_single_updates':int((~events.horizon_complete.astype(bool)).sum())})
    if RUN_SENSITIVITY:
        for gap_ns in sorted({0,100_000,CLUSTER_GAP_NS,500_000}):
            settings=EventConfig(cluster_gap_ns=gap_ns,max_cluster_span_ns=max(MAX_CLUSTER_SPAN_NS,gap_ns),
                reaction_window_ns=REACTION_WINDOW_NS,max_quote_gap_ns=MAX_QUOTE_GAP_NS)
            trial=result if gap_ns==CLUSTER_GAP_NS else detect_events(frame,settings)
            single=single_update_events(trial,settings)
            SENSITIVITY_ROWS.append({'sample':str(key),'cluster_gap_ns':gap_ns,
                'clustered_events':len(trial.events),'single_update_events':len(single),
                'round_trip_clusters':trial.diagnostics['round_trip_clusters'],
                'reversal_clusters':trial.diagnostics['reversal_clusters'],
                'sub_tick_events':int(single.size_class.eq('sub_tick').sum()),
                'single_tick_events':int(single.size_class.eq('single_tick').sum()),
                'multi_tick_events':int(single.size_class.eq('multi_tick').sum()),
                'truncated_events':int((~single.horizon_complete.astype(bool)).sum()),
                'median_observation_duration_ns':float(single.observation_duration_ns.median()) if len(single) else None})
            del trial,single
    # Retain time bounds for the report without holding five days of quote states.
    result.quotes=pd.DataFrame({'ts_event':[result.quotes.ts_event.min(),result.quotes.ts_event.max()]})
    del frame
    if key!='demo': del sample
    print(f'Finished {key}: {len(events):,} primary events',flush=True)
RUN_SUMMARY=pd.DataFrame(RUN_SUMMARIES)
SENSITIVITY=pd.DataFrame(SENSITIVITY_ROWS)
display(RUN_SUMMARY)
if not SENSITIVITY.empty:display(SENSITIVITY)


Analyzing 2026-10-05: 3,646,277 records
Finished 2026-10-05: 46,683 primary events
Analyzing 2026-10-06: 3,669,065 records
Finished 2026-10-06: 35,990 primary events
Analyzing 2026-10-07: 4,547,488 records
Finished 2026-10-07: 61,741 primary events
Analyzing 2026-10-08: 6,148,295 records
Finished 2026-10-08: 113,791 primary events
Analyzing 2026-10-09: 3,299,223 records
Finished 2026-10-09: 44,280 primary events


,sample,source_kind,records,candidates,directional_candidates,clusters,clustered_events,single_update_events,multi_tick_single_updates,truncated_single_updates
0,2026-10-05,market_data,3646277,158413,158413,84620,67544,46683,101,27360
1,2026-10-06,market_data,3669065,141898,141898,71466,55656,35990,44,22129
2,2026-10-07,market_data,4547488,217693,217693,113722,89318,61741,70,36650
3,2026-10-08,market_data,6148295,343533,343533,192992,155551,113791,161,70074
4,2026-10-09,market_data,3299223,150865,150865,80291,64260,44280,52,25769


,sample,cluster_gap_ns,clustered_events,single_update_events,round_trip_clusters,reversal_clusters,sub_tick_events,single_tick_events,multi_tick_events,truncated_events,median_observation_duration_ns
0,2026-10-05,0,158413,158413,0,0,152520,5760,133,119946,273226.0
1,2026-10-05,100000,99709,84471,9195,15627,80761,3602,108,56173,557180.0
2,2026-10-05,250000,67544,46683,7215,17075,44047,2535,101,27360,3520548.0
3,2026-10-05,500000,41465,17831,9430,18361,16148,1594,89,6510,10000000.0
4,2026-10-06,0,141898,141898,0,0,136640,5195,63,109784,252395.0
5,2026-10-06,100000,85021,70055,8567,14879,66745,3261,49,48218,430446.0
6,2026-10-06,250000,55656,35990,6433,15807,33734,2212,44,22129,1422082.0
7,2026-10-06,500000,32722,11897,8705,16791,10441,1413,43,4482,10000000.0
8,2026-10-07,0,217693,217693,0,0,211844,5729,120,167130,264282.0
9,2026-10-07,100000,137705,117620,12500,21384,114260,3285,75,80237,504410.0


In [10]:
DISPLAY_COLUMNS=['event_id','instrument_id','t0','t0_ns','price_before','price_after',
    'direction','move_points','move_ticks','size_class','confirmation_delay_ns',
    'observation_duration_ns','horizon_complete']
for sample,bundle in RESULTS.items():
    print(sample,'— primary single-update events')
    display(bundle['events'].reindex(columns=DISPLAY_COLUMNS).head(20))
    print(sample,'— aggregate paths (secondary specification)')
    display(bundle['result'].events.reindex(columns=DISPLAY_COLUMNS+['cluster_start','cluster_end','n_price_moves']).head(20))
    excluded=bundle['result'].clusters
    counts=excluded.loc[~excluded.selected.astype(bool),'exclusion_reason'].value_counts()
    display(counts.rename_axis('exclusion_reason').rename('clusters').to_frame())


2026-10-05 — primary single-update events


,event_id,instrument_id,t0,t0_ns,price_before,price_after,direction,move_points,move_ticks,size_class,confirmation_delay_ns,observation_duration_ns,horizon_complete
0,0,10252,2026-10-05 13:30:00.000639003+00:00,1791207000000639003,7778.375,7778.250,down,-0.125,-0.5,sub_tick,250000,568892,False
1,4,10252,2026-10-05 13:30:00.012641551+00:00,1791207000012641551,7778.375,7778.500,up,0.125,0.5,sub_tick,250000,296242,False
2,7,10252,2026-10-05 13:30:00.017037293+00:00,1791207000017037293,7778.625,7778.500,down,-0.125,-0.5,sub_tick,250000,301160,False
3,12,10252,2026-10-05 13:30:00.042894321+00:00,1791207000042894321,7778.750,7778.875,up,0.125,0.5,sub_tick,250000,268788,False
4,14,10252,2026-10-05 13:30:00.043719215+00:00,1791207000043719215,7779.125,7779.000,down,-0.125,-0.5,sub_tick,250000,1773500,False
5,15,10252,2026-10-05 13:30:00.045492715+00:00,1791207000045492715,7779.000,7779.125,up,0.125,0.5,sub_tick,250000,1459330,False
6,16,10252,2026-10-05 13:30:00.046952045+00:00,1791207000046952045,7779.125,7779.250,up,0.125,0.5,sub_tick,250000,393930,False
7,17,10252,2026-10-05 13:30:00.047345975+00:00,1791207000047345975,7779.250,7779.375,up,0.125,0.5,sub_tick,250000,10000000,True
8,19,10252,2026-10-05 13:30:00.073367711+00:00,1791207000073367711,7779.625,7779.500,down,-0.125,-0.5,sub_tick,250000,372502,False
9,21,10252,2026-10-05 13:30:00.086437539+00:00,1791207000086437539,7779.375,7779.500,up,0.125,0.5,sub_tick,250000,268578,False


2026-10-05 — aggregate paths (secondary specification)


,event_id,instrument_id,t0,t0_ns,price_before,price_after,direction,move_points,move_ticks,size_class,confirmation_delay_ns,observation_duration_ns,horizon_complete,cluster_start,cluster_end,n_price_moves
0,0,10252,2026-10-05 13:30:00.000639003+00:00,1791207000000639003,7778.375,7778.250,down,-0.125,-0.5,sub_tick,250000,568892,False,2026-10-05 13:30:00.000639003+00:00,2026-10-05 13:30:00.000639003+00:00,1
1,1,10252,2026-10-05 13:30:00.001297995+00:00,1791207000001297995,7778.250,7778.625,up,0.375,1.5,multi_tick,250000,358986,False,2026-10-05 13:30:00.001207895+00:00,2026-10-05 13:30:00.001297995+00:00,3
2,2,10252,2026-10-05 13:30:00.001663431+00:00,1791207000001663431,7778.625,7778.375,down,-0.250,-1.0,single_tick,250000,2509332,False,2026-10-05 13:30:00.001656981+00:00,2026-10-05 13:30:00.001663431+00:00,2
3,4,10252,2026-10-05 13:30:00.012641551+00:00,1791207000012641551,7778.375,7778.500,up,0.125,0.5,sub_tick,250000,296242,False,2026-10-05 13:30:00.012641551+00:00,2026-10-05 13:30:00.012641551+00:00,1
4,7,10252,2026-10-05 13:30:00.017037293+00:00,1791207000017037293,7778.625,7778.500,down,-0.125,-0.5,sub_tick,250000,301160,False,2026-10-05 13:30:00.017037293+00:00,2026-10-05 13:30:00.017037293+00:00,1
5,10,10252,2026-10-05 13:30:00.018866343+00:00,1791207000018866343,7778.625,7778.875,up,0.250,1.0,single_tick,250000,10000000,True,2026-10-05 13:30:00.018824449+00:00,2026-10-05 13:30:00.018866343+00:00,2
6,12,10252,2026-10-05 13:30:00.042894321+00:00,1791207000042894321,7778.750,7778.875,up,0.125,0.5,sub_tick,250000,268788,False,2026-10-05 13:30:00.042894321+00:00,2026-10-05 13:30:00.042894321+00:00,1
7,13,10252,2026-10-05 13:30:00.043176913+00:00,1791207000043176913,7778.875,7779.125,up,0.250,1.0,single_tick,250000,542302,False,2026-10-05 13:30:00.043163109+00:00,2026-10-05 13:30:00.043176913+00:00,2
8,14,10252,2026-10-05 13:30:00.043719215+00:00,1791207000043719215,7779.125,7779.000,down,-0.125,-0.5,sub_tick,250000,1773500,False,2026-10-05 13:30:00.043719215+00:00,2026-10-05 13:30:00.043719215+00:00,1
9,15,10252,2026-10-05 13:30:00.045492715+00:00,1791207000045492715,7779.000,7779.125,up,0.125,0.5,sub_tick,250000,1459330,False,2026-10-05 13:30:00.045492715+00:00,2026-10-05 13:30:00.045492715+00:00,1


,clusters
exclusion_reason,
reversal,9851
round_trip;reversal,7211
reversal;span_cap_split,9
round_trip;reversal;span_cap_split,4
span_cap_split,1


...

2026-10-09 — primary single-update events


,event_id,instrument_id,t0,t0_ns,price_before,price_after,direction,move_points,move_ticks,size_class,confirmation_delay_ns,observation_duration_ns,horizon_complete
0,0,10252,2026-10-09 13:30:00.002581589+00:00,1791552600002581589,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,274700,False
1,1,10252,2026-10-09 13:30:00.002856289+00:00,1791552600002856289,7838.250,7838.125,down,-0.125,-0.5,sub_tick,250000,1204978,False
2,3,10252,2026-10-09 13:30:00.004889423+00:00,1791552600004889423,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,259860,False
3,6,10252,2026-10-09 13:30:00.006252201+00:00,1791552600006252201,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,258668,False
4,9,10252,2026-10-09 13:30:00.008837731+00:00,1791552600008837731,7838.125,7838.375,up,0.250,1.0,single_tick,250000,735634,False
5,12,10252,2026-10-09 13:30:00.021416137+00:00,1791552600021416137,7838.125,7838.000,down,-0.125,-0.5,sub_tick,250000,551094,False
6,13,10252,2026-10-09 13:30:00.021967231+00:00,1791552600021967231,7838.000,7837.875,down,-0.125,-0.5,sub_tick,250000,1208106,False
7,15,10252,2026-10-09 13:30:00.028405727+00:00,1791552600028405727,7838.125,7838.375,up,0.250,1.0,single_tick,250000,10000000,True
8,16,10252,2026-10-09 13:30:00.045245369+00:00,1791552600045245369,7838.375,7838.500,up,0.125,0.5,sub_tick,250000,308474,False
9,17,10252,2026-10-09 13:30:00.045553843+00:00,1791552600045553843,7838.500,7838.625,up,0.125,0.5,sub_tick,250000,6468906,False


2026-10-09 — aggregate paths (secondary specification)


,event_id,instrument_id,t0,t0_ns,price_before,price_after,direction,move_points,move_ticks,size_class,confirmation_delay_ns,observation_duration_ns,horizon_complete,cluster_start,cluster_end,n_price_moves
0,0,10252,2026-10-09 13:30:00.002581589+00:00,1791552600002581589,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,274700,False,2026-10-09 13:30:00.002581589+00:00,2026-10-09 13:30:00.002581589+00:00,1
1,1,10252,2026-10-09 13:30:00.002856289+00:00,1791552600002856289,7838.250,7838.125,down,-0.125,-0.5,sub_tick,250000,1204978,False,2026-10-09 13:30:00.002856289+00:00,2026-10-09 13:30:00.002856289+00:00,1
2,3,10252,2026-10-09 13:30:00.004889423+00:00,1791552600004889423,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,259860,False,2026-10-09 13:30:00.004889423+00:00,2026-10-09 13:30:00.004889423+00:00,1
3,6,10252,2026-10-09 13:30:00.006252201+00:00,1791552600006252201,7838.125,7838.250,up,0.125,0.5,sub_tick,250000,258668,False,2026-10-09 13:30:00.006252201+00:00,2026-10-09 13:30:00.006252201+00:00,1
4,9,10252,2026-10-09 13:30:00.008837731+00:00,1791552600008837731,7838.125,7838.375,up,0.250,1.0,single_tick,250000,735634,False,2026-10-09 13:30:00.008837731+00:00,2026-10-09 13:30:00.008837731+00:00,1
5,10,10252,2026-10-09 13:30:00.009785901+00:00,1791552600009785901,7838.375,7838.125,down,-0.250,-1.0,single_tick,250000,250184,False,2026-10-09 13:30:00.009573365+00:00,2026-10-09 13:30:00.009785901+00:00,2
6,12,10252,2026-10-09 13:30:00.021416137+00:00,1791552600021416137,7838.125,7838.000,down,-0.125,-0.5,sub_tick,250000,551094,False,2026-10-09 13:30:00.021416137+00:00,2026-10-09 13:30:00.021416137+00:00,1
7,13,10252,2026-10-09 13:30:00.021967231+00:00,1791552600021967231,7838.000,7837.875,down,-0.125,-0.5,sub_tick,250000,1208106,False,2026-10-09 13:30:00.021967231+00:00,2026-10-09 13:30:00.021967231+00:00,1
8,14,10252,2026-10-09 13:30:00.023177797+00:00,1791552600023177797,7837.875,7838.125,up,0.250,1.0,single_tick,250000,5227930,False,2026-10-09 13:30:00.023175337+00:00,2026-10-09 13:30:00.023177797+00:00,2
9,15,10252,2026-10-09 13:30:00.028405727+00:00,1791552600028405727,7838.125,7838.375,up,0.250,1.0,single_tick,250000,10000000,True,2026-10-09 13:30:00.028405727+00:00,2026-10-09 13:30:00.028405727+00:00,1


,clusters
exclusion_reason,
reversal,9299
round_trip;reversal,6713
reversal;span_cap_split,12
round_trip;reversal;span_cap_split,4
span_cap_split,3


## Save the results

Large tables use compressed Parquet, which keeps UTC timestamps and exact
nanosecond integers. Small summary tables use CSV. Each date has its own folder;
`PR_summary.md` combines the run settings, actual counts, tests and limitations
in one file for review. Re-running overwrites the outputs for the selected dates.

`events.parquet` is the primary trigger table. `clustered_events.parquet` contains
net moves over paths and should be treated as a separate specification.


In [11]:
import hashlib
import importlib.metadata
import platform

OUT_DIR.mkdir(parents=True, exist_ok=True)

def md_table(table):
    """Small tables only; full event tables stay in Parquet."""
    if table.empty:
        return 'No rows.'
    def cell(value):
        return str(value).replace('|', r'\|').replace('\n', ' ')
    lines = ['| ' + ' | '.join(map(cell, table.columns)) + ' |',
             '| ' + ' | '.join(['---'] * len(table.columns)) + ' |']
    lines.extend('| ' + ' | '.join(map(cell, row)) + ' |'
                 for row in table.itertuples(index=False, name=None))
    return '\n'.join(lines)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

versions = {p: importlib.metadata.version(p)
            for p in ['pandas', 'numpy', 'databento', 'pyarrow', 'pytest']}
EXPORTS = {}
REPORT_SECTIONS = []
ALL_CLASS_COUNTS = []
for sample, bundle in RESULTS.items():
    result, events, source = bundle['result'], bundle['events'], SOURCES[sample]
    folder = OUT_DIR / sample
    folder.mkdir(parents=True, exist_ok=True)
    tables = {'candidates': result.candidates, 'clusters': result.clusters,
              'clustered_events': result.events, 'events': events}
    file_rows = []
    for name, table in tables.items():
        target = folder / f'{name}.parquet'
        temporary = target.with_suffix('.partial.parquet')
        table.to_parquet(temporary, index=False, compression='zstd')
        temporary.replace(target)
        # Check exact timestamp preservation in the primary output.
        if name == 'events':
            saved = pd.read_parquet(target, columns=['t0', 't0_ns', 'move_ticks'])
            pd.testing.assert_series_equal(saved.t0, events.t0, check_names=False)
            pd.testing.assert_series_equal(saved.t0_ns, events.t0_ns, check_names=False)
            assert (saved.t0.astype('int64').to_numpy() == saved.t0_ns.to_numpy(dtype='int64')).all()
            assert np.array_equal(saved.move_ticks.to_numpy(), events.move_ticks.to_numpy())
        file_rows.append({'file': f'{sample}/{target.name}', 'rows': len(table),
                          'MiB': round(target.stat().st_size / 1024**2, 3)})
    file_table = pd.DataFrame(file_rows)
    class_counts = (events.groupby(['direction', 'size_class'], observed=True)
                    .size().rename('events').reset_index())
    class_counts.insert(0, 'sample', sample)
    ALL_CLASS_COUNTS.append(class_counts)
    excluded = result.clusters.loc[~result.clusters.selected.astype(bool), 'exclusion_reason']
    # A cluster may have several reasons, so these counts can overlap.
    reason_counts = (excluded.str.split(';').explode().value_counts()
                     .rename_axis('reason').rename('clusters').reset_index())
    sensitivity = SENSITIVITY.loc[SENSITIVITY['sample'].eq(sample)].copy() if not SENSITIVITY.empty else SENSITIVITY
    class_counts.to_csv(folder / 'event_counts.csv', index=False)
    reason_counts.to_csv(folder / 'exclusion_counts.csv', index=False)
    sensitivity.to_csv(folder / 'filter_sensitivity.csv', index=False)
    file_table.to_csv(folder / 'output_inventory.csv', index=False)
    provenance = dict(source)
    if source['kind'] == 'market_data':
        provenance['path'] = str(Path(source['path']).relative_to(ROOT))
        provenance['input_sha256'] = sha256_file(source['path'])
    diagnostics = {**result.diagnostics, 'time_unit': 'ns',
                   'primary_event_definition': 'single_price_update',
                   'primary_events': len(events), 'source': provenance,
                   'test_results': {'passed': TEST_RESULTS.passed, 'failed': TEST_RESULTS.failed,
                                    'skipped': TEST_RESULTS.skipped, 'exit_code': TEST_EXIT_CODE},
                   'versions': versions, 'python': platform.python_version(),
                   'parquet_timestamp_roundtrip': 'passed'}
    (folder / 'diagnostics.json').write_text(json.dumps(diagnostics, indent=2, default=str), encoding='utf-8')
    if len(events):
        # These checks concern the selected sample, not every raw move.
        assert events.t0.is_monotonic_increasing and events.t0.is_unique
        assert np.allclose(events.move_points / SETTINGS.tick_size, events.move_ticks)
        assert events.n_price_moves.eq(1).all()
        assert (events.observation_end_ns > events.t0_ns).all()
        if len(events)>1:
            assert (events.observation_end_ns.iloc[:-1].to_numpy() <= events.t0_ns.iloc[1:].to_numpy()).all()
    primary_count = len(events)
    truncated = int((~events.horizon_complete.astype(bool)).sum())
    median_window = int(events.observation_duration_ns.median()) if primary_count else 'N/A'
    manifest = source.get('manifest', {})
    sample_description = (f"{manifest.get('raw_symbols', ['unknown'])[0]}, instrument ID "
                          f"{manifest.get('symbols', ['unknown'])[0]}; requested interval "
                          f"{manifest.get('start')} to {manifest.get('end')}."
                          if source['kind']=='market_data' else 'Synthetic example; not a market-data result.')
    q = result.quotes
    section = f'''## Sample: {sample}

{sample_description}

| Measure | Count |
| --- | ---: |
| Input records | {result.diagnostics['input_records']:,} |
| Trade records ignored | {result.diagnostics['trade_records_ignored']:,} |
| Same-timestamp quote records collapsed | {result.diagnostics['same_timestamp_records_collapsed']:,} |
| Invalid final quote states | {result.diagnostics['invalid_states']:,} |
| Continuity segments | {result.diagnostics['continuity_segments']:,} |
| BBO price-change candidates | {len(result.candidates):,} |
| Midpoint-changing candidates | {result.diagnostics['directional_candidates']:,} |
| Clusters before filtering | {len(result.clusters):,} |
| Selected paths, including single updates | {len(result.events):,} |
| Primary single-update events | {primary_count:,} |
| Primary events with truncated horizons | {truncated:,} |

Observed final quote-state timestamps: {q.ts_event.min()} to {q.ts_event.max()}.
The median primary observation window is {median_window} ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

{md_table(class_counts)}

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

{md_table(reason_counts)}

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

{md_table(sensitivity)}

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

{md_table(file_table)}

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.
'''
    (folder / 'sample_summary.md').write_text(section, encoding='utf-8')
    REPORT_SECTIONS.append(section)
    EXPORTS[sample] = folder

RUN_SUMMARY.to_csv(OUT_DIR / 'run_summary.csv', index=False)
SENSITIVITY.to_csv(OUT_DIR / 'filter_sensitivity.csv', index=False)
pd.concat(ALL_CLASS_COUNTS, ignore_index=True).to_csv(OUT_DIR / 'event_counts.csv', index=False)
(OUT_DIR / 'requirements-validated.txt').write_text(
    '# Python ' + platform.python_version() + '\n' + '\n'.join(f'{p}=={v}' for p,v in versions.items()) + '\n', encoding='utf-8')
real_run = all(s['kind']=='market_data' for s in SOURCES.values())
report = f'''# Issue 4: Detect and classify ES futures price-move events

## What this adds

A self-contained notebook reads the Issue 3 futures cache and exports reproducible
price-move events for the stale-quote study. It keeps candidates and cluster
exclusions so we can review what the filter removes.

Run type: **{'real cached market data' if real_run else 'synthetic demo; not empirical validation'}**.
Offline tests: **{TEST_RESULTS.passed} passed, {TEST_RESULTS.failed} failed, {TEST_RESULTS.skipped} skipped**.
The saved primary Parquet table also passed exact timestamp and signed-tick
round-trip checks. Selected primary event times are unique, move sizes agree
with prices, and their observation windows do not overlap.

## Event definition

We use valid best bid/offer states from `mbp-1`. Trade records are ignored because
their BBO is pre-trade. Size-only changes do not trigger an event. For multiple
quote records with one exchange timestamp, we keep the final state before
calculating changes. This is a timestamp-level state definition: moves reversed
within that same timestamp are not separately observable in this table.
Invalid states, resets and quote gaps over {MAX_QUOTE_GAP_NS:,} ns break continuity.
We do not compare prices across those breaks or between different futures.
The future is taken from `futures.json`, rather than re-selected from a roll rule.

The primary price is `(bid + ask) / 2`. One ES tick is {SETTINGS.tick_size} index
points. We retain signed point/tick changes, classify up/down and distinguish
sub-tick, single-tick and multi-tick moves. A native single update can be multi-tick.
The primary table contains one midpoint-changing update per selected cluster;
`t0` is that update's exchange `ts_event`. All times are UTC. Epoch timestamps
and durations are stored in integer ns, without conversion to microseconds.
Nanosecond representation alone does not validate cross-contract clock alignment.

## Clustering, bouncing and overlap

Adjacent BBO price changes join a cluster when their gap is at most
{CLUSTER_GAP_NS:,} ns and the total span from the first member is at most
{MAX_CLUSTER_SPAN_NS:,} ns. Round trips and partial reversals are excluded.
Clusters linked across a span-cap split are excluded, so the cap does not create
artificial independent moves. Quiet context is required on both sides.

`clustered_events.parquet` stores selected net paths as a secondary specification;
its `t0` is the final constituent update, never the first one. `confirmed_at` is
cluster end plus the gap. Filtering uses later quotes and is retrospective.
It should not be described as an immediately available trading signal.

The observation window is `[t0, observation_end)`. It ends at the next raw
midpoint move, the {REACTION_WINDOW_NS:,} ns horizon or the segment boundary,
whichever occurs first. A later move ends the window even if that move is
filtered out. A truncated window is recorded, not treated as an option response.

''' + '\n\n'.join(REPORT_SECTIONS) + f'''
## Validation and scope

The offline suite covers direction/tick classification, half ticks, same-timestamp
updates, invalid states, bounded clusters, bouncing, overlap, exact ns boundaries,
manifest checks and compressed DBN decoding. The full cached sample was processed
with the settings above. Input checksums, parameters and dependency versions are
saved in each date's `diagnostics.json`.

{'The counts above are from the supplied trading day. More dates and different filter thresholds remain robustness checks.' if real_run else 'Re-run with real cached data before reporting empirical counts.'}
The filtered events are a selected population, not all futures moves. This step
does not establish timestamp comparability with options or estimate stale-quote
profitability. Those questions remain for the downstream analysis.

## Reproduce and review

Place `issue4_engine.ipynb` in the project root (or `notebooks/`). Keep the Issue 3
cache at `data/raw/mbp-1/YYYYMMDD/futures.dbn.zst` with `futures.json`. Use Python
3.11+ and Run All with `RUN_MODE = 'real'`. Missing dependencies are installed in
the current kernel. The notebook makes no data-download requests. Outputs go to
`issue4_report/`; date folders are regenerated for the dates selected in the run.

Submit the executed notebook and this Markdown report. The large Parquet files
can remain local for subsequent analysis. CSV summaries and `sample_summary.md`
repeat the tables above and need not also be included in the PR. Raw DBN files
should stay out of Git. No API key is used or saved.
'''
(OUT_DIR / 'PR_summary.md').write_text(report, encoding='utf-8')
display(Markdown(report))
print('Saved reports and tables to:', OUT_DIR)


# Issue 4: Detect and classify ES futures price-move events

## What this adds

A self-contained notebook reads the Issue 3 futures cache and exports reproducible
price-move events for the stale-quote study. It keeps candidates and cluster
exclusions so we can review what the filter removes.

Run type: **real cached market data**.
Offline tests: **68 passed, 0 failed, 0 skipped**.
The saved primary Parquet table also passed exact timestamp and signed-tick
round-trip checks. Selected primary event times are unique, move sizes agree
with prices, and their observation windows do not overlap.

## Event definition

We use valid best bid/offer states from `mbp-1`. Trade records are ignored because
their BBO is pre-trade. Size-only changes do not trigger an event. For multiple
quote records with one exchange timestamp, we keep the final state before
calculating changes. This is a timestamp-level state definition: moves reversed
within that same timestamp are not separately observable in this table.
Invalid states, resets and quote gaps over 1,000,000,000 ns break continuity.
We do not compare prices across those breaks or between different futures.
The future is taken from `futures.json`, rather than re-selected from a roll rule.

The primary price is `(bid + ask) / 2`. One ES tick is 0.25 index
points. We retain signed point/tick changes, classify up/down and distinguish
sub-tick, single-tick and multi-tick moves. A native single update can be multi-tick.
The primary table contains one midpoint-changing update per selected cluster;
`t0` is that update's exchange `ts_event`. All times are UTC. Epoch timestamps
and durations are stored in integer ns, without conversion to microseconds.
Nanosecond representation alone does not validate cross-contract clock alignment.

## Clustering, bouncing and overlap

Adjacent BBO price changes join a cluster when their gap is at most
250,000 ns and the total span from the first member is at most
2,000,000 ns. Round trips and partial reversals are excluded.
Clusters linked across a span-cap split are excluded, so the cap does not create
artificial independent moves. Quiet context is required on both sides.

`clustered_events.parquet` stores selected net paths as a secondary specification;
its `t0` is the final constituent update, never the first one. `confirmed_at` is
cluster end plus the gap. Filtering uses later quotes and is retrospective.
It should not be described as an immediately available trading signal.

The observation window is `[t0, observation_end)`. It ends at the next raw
midpoint move, the 10,000,000 ns horizon or the segment boundary,
whichever occurs first. A later move ends the window even if that move is
filtered out. A truncated window is recorded, not treated as an option response.

## Sample: 2026-10-05

ESZ6, instrument ID 10252; requested interval 2026-10-05 09:30:00-04:00 to 2026-10-05 16:00:00-04:00.

| Measure | Count |
| --- | ---: |
| Input records | 3,646,277 |
| Trade records ignored | 255,212 |
| Same-timestamp quote records collapsed | 27 |
| Invalid final quote states | 0 |
| Continuity segments | 89 |
| BBO price-change candidates | 158,413 |
| Midpoint-changing candidates | 158,413 |
| Clusters before filtering | 84,620 |
| Selected paths, including single updates | 67,544 |
| Primary single-update events | 46,683 |
| Primary events with truncated horizons | 27,360 |

Observed final quote-state timestamps: 2026-10-05 13:29:59.999951635+00:00 to 2026-10-05 19:59:59.999851975+00:00.
The median primary observation window is 3520548 ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

| sample | direction | size_class | events |
| --- | --- | --- | --- |
| 2026-10-05 | down | multi_tick | 47 |
| 2026-10-05 | down | single_tick | 1168 |
| 2026-10-05 | down | sub_tick | 22170 |
| 2026-10-05 | up | multi_tick | 54 |
| 2026-10-05 | up | single_tick | 1367 |
| 2026-10-05 | up | sub_tick | 21877 |

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

| reason | clusters |
| --- | --- |
| reversal | 17075 |
| round_trip | 7215 |
| span_cap_split | 14 |

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

| sample | cluster_gap_ns | clustered_events | single_update_events | round_trip_clusters | reversal_clusters | sub_tick_events | single_tick_events | multi_tick_events | truncated_events | median_observation_duration_ns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-10-05 | 0 | 158413 | 158413 | 0 | 0 | 152520 | 5760 | 133 | 119946 | 273226.0 |
| 2026-10-05 | 100000 | 99709 | 84471 | 9195 | 15627 | 80761 | 3602 | 108 | 56173 | 557180.0 |
| 2026-10-05 | 250000 | 67544 | 46683 | 7215 | 17075 | 44047 | 2535 | 101 | 27360 | 3520548.0 |
| 2026-10-05 | 500000 | 41465 | 17831 | 9430 | 18361 | 16148 | 1594 | 89 | 6510 | 10000000.0 |

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

| file | rows | MiB |
| --- | --- | --- |
| 2026-10-05/candidates.parquet | 158413 | 3.4 |
| 2026-10-05/clusters.parquet | 84620 | 8.262 |
| 2026-10-05/clustered_events.parquet | 67544 | 7.085 |
| 2026-10-05/events.parquet | 46683 | 4.851 |

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.


## Sample: 2026-10-06

ESZ6, instrument ID 10252; requested interval 2026-10-06 09:30:00-04:00 to 2026-10-06 16:00:00-04:00.

| Measure | Count |
| --- | ---: |
| Input records | 3,669,065 |
| Trade records ignored | 243,713 |
| Same-timestamp quote records collapsed | 35 |
| Invalid final quote states | 0 |
| Continuity segments | 77 |
| BBO price-change candidates | 141,898 |
| Midpoint-changing candidates | 141,898 |
| Clusters before filtering | 71,466 |
| Selected paths, including single updates | 55,656 |
| Primary single-update events | 35,990 |
| Primary events with truncated horizons | 22,129 |

Observed final quote-state timestamps: 2026-10-06 13:30:00.000012823+00:00 to 2026-10-06 19:59:59.999263543+00:00.
The median primary observation window is 1422082 ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

| sample | direction | size_class | events |
| --- | --- | --- | --- |
| 2026-10-06 | down | multi_tick | 26 |
| 2026-10-06 | down | single_tick | 1099 |
| 2026-10-06 | down | sub_tick | 16325 |
| 2026-10-06 | up | multi_tick | 18 |
| 2026-10-06 | up | single_tick | 1113 |
| 2026-10-06 | up | sub_tick | 17409 |

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

| reason | clusters |
| --- | --- |
| reversal | 15807 |
| round_trip | 6433 |
| span_cap_split | 16 |
| missing_left_context | 1 |

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

| sample | cluster_gap_ns | clustered_events | single_update_events | round_trip_clusters | reversal_clusters | sub_tick_events | single_tick_events | multi_tick_events | truncated_events | median_observation_duration_ns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-10-06 | 0 | 141898 | 141898 | 0 | 0 | 136640 | 5195 | 63 | 109784 | 252395.0 |
| 2026-10-06 | 100000 | 85021 | 70055 | 8567 | 14879 | 66745 | 3261 | 49 | 48218 | 430446.0 |
| 2026-10-06 | 250000 | 55656 | 35990 | 6433 | 15807 | 33734 | 2212 | 44 | 22129 | 1422082.0 |
| 2026-10-06 | 500000 | 32722 | 11897 | 8705 | 16791 | 10441 | 1413 | 43 | 4482 | 10000000.0 |

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

| file | rows | MiB |
| --- | --- | --- |
| 2026-10-06/candidates.parquet | 141898 | 2.896 |
| 2026-10-06/clusters.parquet | 71466 | 7.34 |
| 2026-10-06/clustered_events.parquet | 55656 | 5.902 |
| 2026-10-06/events.parquet | 35990 | 3.783 |

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.


## Sample: 2026-10-07

ESZ6, instrument ID 10252; requested interval 2026-10-07 09:30:00-04:00 to 2026-10-07 16:00:00-04:00.

| Measure | Count |
| --- | ---: |
| Input records | 4,547,488 |
| Trade records ignored | 273,162 |
| Same-timestamp quote records collapsed | 19 |
| Invalid final quote states | 0 |
| Continuity segments | 44 |
| BBO price-change candidates | 217,693 |
| Midpoint-changing candidates | 217,693 |
| Clusters before filtering | 113,722 |
| Selected paths, including single updates | 89,318 |
| Primary single-update events | 61,741 |
| Primary events with truncated horizons | 36,650 |

Observed final quote-state timestamps: 2026-10-07 13:30:00.000015857+00:00 to 2026-10-07 19:59:59.999837823+00:00.
The median primary observation window is 3244946 ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

| sample | direction | size_class | events |
| --- | --- | --- | --- |
| 2026-10-07 | down | multi_tick | 30 |
| 2026-10-07 | down | single_tick | 1120 |
| 2026-10-07 | down | sub_tick | 29820 |
| 2026-10-07 | up | multi_tick | 40 |
| 2026-10-07 | up | single_tick | 1012 |
| 2026-10-07 | up | sub_tick | 29719 |

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

| reason | clusters |
| --- | --- |
| reversal | 24401 |
| round_trip | 10411 |
| span_cap_split | 33 |
| missing_left_context | 1 |

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

| sample | cluster_gap_ns | clustered_events | single_update_events | round_trip_clusters | reversal_clusters | sub_tick_events | single_tick_events | multi_tick_events | truncated_events | median_observation_duration_ns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-10-07 | 0 | 217693 | 217693 | 0 | 0 | 211844 | 5729 | 120 | 167130 | 264282.0 |
| 2026-10-07 | 100000 | 137705 | 117620 | 12500 | 21384 | 114260 | 3285 | 75 | 80237 | 504410.0 |
| 2026-10-07 | 250000 | 89318 | 61741 | 10411 | 24401 | 59539 | 2132 | 70 | 36650 | 3244946.0 |
| 2026-10-07 | 500000 | 54641 | 25384 | 13705 | 26268 | 24073 | 1253 | 58 | 9332 | 10000000.0 |

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

| file | rows | MiB |
| --- | --- | --- |
| 2026-10-07/candidates.parquet | 217693 | 3.919 |
| 2026-10-07/clusters.parquet | 113722 | 11.274 |
| 2026-10-07/clustered_events.parquet | 89318 | 9.43 |
| 2026-10-07/events.parquet | 61741 | 6.383 |

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.


## Sample: 2026-10-08

ESZ6, instrument ID 10252; requested interval 2026-10-08 09:30:00-04:00 to 2026-10-08 16:00:00-04:00.

| Measure | Count |
| --- | ---: |
| Input records | 6,148,295 |
| Trade records ignored | 361,053 |
| Same-timestamp quote records collapsed | 66 |
| Invalid final quote states | 0 |
| Continuity segments | 8 |
| BBO price-change candidates | 343,533 |
| Midpoint-changing candidates | 343,533 |
| Clusters before filtering | 192,992 |
| Selected paths, including single updates | 155,551 |
| Primary single-update events | 113,791 |
| Primary events with truncated horizons | 70,074 |

Observed final quote-state timestamps: 2026-10-08 13:30:00.000014289+00:00 to 2026-10-08 19:59:59.999747485+00:00.
The median primary observation window is 3863984 ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

| sample | direction | size_class | events |
| --- | --- | --- | --- |
| 2026-10-08 | down | multi_tick | 85 |
| 2026-10-08 | down | single_tick | 1789 |
| 2026-10-08 | down | sub_tick | 54891 |
| 2026-10-08 | up | multi_tick | 76 |
| 2026-10-08 | up | single_tick | 1631 |
| 2026-10-08 | up | sub_tick | 55319 |

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

| reason | clusters |
| --- | --- |
| reversal | 37439 |
| round_trip | 16936 |
| span_cap_split | 20 |
| missing_left_context | 1 |

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

| sample | cluster_gap_ns | clustered_events | single_update_events | round_trip_clusters | reversal_clusters | sub_tick_events | single_tick_events | multi_tick_events | truncated_events | median_observation_duration_ns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-10-08 | 0 | 343533 | 343533 | 0 | 0 | 335792 | 7485 | 256 | 261328 | 293190.0 |
| 2026-10-08 | 100000 | 231843 | 204153 | 18806 | 30884 | 199203 | 4767 | 183 | 140588 | 662370.0 |
| 2026-10-08 | 250000 | 155551 | 113791 | 16936 | 37439 | 110210 | 3420 | 161 | 70074 | 3863984.0 |
| 2026-10-08 | 500000 | 96978 | 49871 | 22557 | 42585 | 47591 | 2150 | 130 | 21585 | 10000000.0 |

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

| file | rows | MiB |
| --- | --- | --- |
| 2026-10-08/candidates.parquet | 343533 | 6.098 |
| 2026-10-08/clusters.parquet | 192992 | 18.153 |
| 2026-10-08/clustered_events.parquet | 155551 | 15.538 |
| 2026-10-08/events.parquet | 113791 | 9.601 |

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.


## Sample: 2026-10-09

ESZ6, instrument ID 10252; requested interval 2026-10-09 09:30:00-04:00 to 2026-10-09 16:00:00-04:00.

| Measure | Count |
| --- | ---: |
| Input records | 3,299,223 |
| Trade records ignored | 214,265 |
| Same-timestamp quote records collapsed | 47 |
| Invalid final quote states | 0 |
| Continuity segments | 121 |
| BBO price-change candidates | 150,865 |
| Midpoint-changing candidates | 150,865 |
| Clusters before filtering | 80,291 |
| Selected paths, including single updates | 64,260 |
| Primary single-update events | 44,280 |
| Primary events with truncated horizons | 25,769 |

Observed final quote-state timestamps: 2026-10-09 13:30:00.000012951+00:00 to 2026-10-09 19:59:59.999852525+00:00.
The median primary observation window is 3662065 ns. This is an available
futures observation window, not an option reaction-time estimate.

### Direction and move size

| sample | direction | size_class | events |
| --- | --- | --- | --- |
| 2026-10-09 | down | multi_tick | 29 |
| 2026-10-09 | down | single_tick | 894 |
| 2026-10-09 | down | sub_tick | 20527 |
| 2026-10-09 | up | multi_tick | 23 |
| 2026-10-09 | up | single_tick | 1048 |
| 2026-10-09 | up | sub_tick | 21759 |

Move ticks are signed: negative means down. The size label uses absolute ticks.
Midpoints can move by half a futures tick.

### Excluded clusters

| reason | clusters |
| --- | --- |
| reversal | 16028 |
| round_trip | 6717 |
| span_cap_split | 19 |

Reasons can overlap; their counts should not be added to obtain a total.
The primary table also omits selected paths with more than one midpoint move.

### Clustering-gap sensitivity

| sample | cluster_gap_ns | clustered_events | single_update_events | round_trip_clusters | reversal_clusters | sub_tick_events | single_tick_events | multi_tick_events | truncated_events | median_observation_duration_ns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-10-09 | 0 | 150865 | 150865 | 0 | 0 | 146242 | 4515 | 108 | 114053 | 271786.0 |
| 2026-10-09 | 100000 | 96697 | 82676 | 8335 | 14338 | 79806 | 2812 | 58 | 55359 | 527391.0 |
| 2026-10-09 | 250000 | 64260 | 44280 | 6717 | 16028 | 42286 | 1942 | 52 | 25769 | 3662065.0 |
| 2026-10-09 | 500000 | 39445 | 17946 | 9394 | 17814 | 16702 | 1198 | 46 | 6435 | 10000000.0 |

Other parameters and the input sample are held fixed. The zero-gap case still
uses the same final-state, validity and boundary rules. These samples differ in
composition; a higher event count alone does not establish a better filter.

### Generated tables

| file | rows | MiB |
| --- | --- | --- |
| 2026-10-09/candidates.parquet | 150865 | 2.996 |
| 2026-10-09/clusters.parquet | 80291 | 8.371 |
| 2026-10-09/clustered_events.parquet | 64260 | 6.761 |
| 2026-10-09/events.parquet | 44280 | 4.605 |

The Parquet tables are local analysis outputs. Running the notebook regenerates
them; they are not required attachments to this PR.

## Validation and scope

The offline suite covers direction/tick classification, half ticks, same-timestamp
updates, invalid states, bounded clusters, bouncing, overlap, exact ns boundaries,
manifest checks and compressed DBN decoding. The full cached sample was processed
with the settings above. Input checksums, parameters and dependency versions are
saved in each date's `diagnostics.json`.

The counts above are from the supplied trading day. More dates and different filter thresholds remain robustness checks.
The filtered events are a selected population, not all futures moves. This step
does not establish timestamp comparability with options or estimate stale-quote
profitability. Those questions remain for the downstream analysis.

## Reproduce and review

Place `issue4_engine.ipynb` in the project root (or `notebooks/`). Keep the Issue 3
cache at `data/raw/mbp-1/YYYYMMDD/futures.dbn.zst` with `futures.json`. Use Python
3.11+ and Run All with `RUN_MODE = 'real'`. Missing dependencies are installed in
the current kernel. The notebook makes no data-download requests. Outputs go to
`issue4_report/`; date folders are regenerated for the dates selected in the run.

Submit the executed notebook and this Markdown report. The large Parquet files
can remain local for subsequent analysis. CSV summaries and `sample_summary.md`
repeat the tables above and need not also be included in the PR. Raw DBN files
should stay out of Git. No API key is used or saved.


Saved reports and tables to: /Users/christina/Desktop/Futures/reports/Futures_Project-issue-3-download/issue4_report
